# Multi-Factor Regression Model

A quick look at how much of an asset's return can be explained by well-known systematic risk factors, using the Fama-French 5 factors (Market, SMB, HML, RMW, CMA) plus Momentum (6 total).

- **Static regression**: one set of factor loadings (betas) over the full sample
- **Rolling regression**: how those loadings drift over time, and how much of the return the model explains (R²) in each window

Assets: `VTI` (total market), `VLUE` (value ETF), `MTUM` (momentum ETF), and `AAPL` (a single stock, included as a less "factor-tilted" comparison point).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm
import getFamaFrenchFactors as gff

## Data

In [ ]:
rets = yf.download(['VTI','VLUE','MTUM','AAPL'], interval = '1mo', auto_adjust = True, start = '2000-01-01')['Close']
rets = rets.to_period('M').pct_change().dropna()

In [ ]:
rets.tail()

In [ ]:
ff_5f = gff.famaFrench5Factor().set_index('date_ff_factors').to_period('M')
ff_momentum = gff.momentumFactor().set_index('date_ff_factors').to_period('M')

In [ ]:
ff_6f = ff_5f.join(ff_momentum)
ff_6f = ff_6f[[*ff_6f.columns.drop('RF'),'RF']]
(1+ff_6f).cumprod().plot(figsize=(12,6), title='Cumulative Growth of Fama-French 6 Factors')

## Full-Sample Factor Regression

In [ ]:
def factor_regression(rets, factor_model):
    # common index
    common_idx = rets.index.intersection(factor_model.index)
    rets = rets.loc[common_idx]
    factor_model = factor_model.loc[common_idx]

    factor_cols = factor_model.columns.drop('RF')

    # prep data for regression
    rets_excess = rets - factor_model[['RF']].values
    X = factor_model[factor_cols].copy()
    X['Alpha'] = 1

    # run regression
    lm = sm.OLS(rets_excess, X).fit()
    factor_loadings = lm.params
    factor_loadings.columns = rets.columns

    # calculate annualized alpha
    alpha_per_period = factor_loadings.loc[['Alpha']]
    annualized_alpha = ((1+alpha_per_period)**12)-1
    annualized_alpha.index = ['Annualized Alpha']
    factor_loadings = pd.concat([factor_loadings, annualized_alpha])

    # alpha & annualized alpha in %
    factor_loadings = factor_loadings.round(2)
    factor_loadings = factor_loadings.astype(object)
    factor_loadings.loc[['Alpha']] = (alpha_per_period * 100).round(2).astype(str) + '%'
    factor_loadings.loc[['Annualized Alpha']] = (annualized_alpha * 100).round(2).astype(str) + '%'

    return factor_loadings

In [ ]:
factor_regression(rets, ff_6f)

**Note:** `VLUE` and `MTUM` are themselves factor-tilted ETFs, so it's not surprising if they load heavily on HML and MOM respectively — almost by construction. `AAPL` is included as a plain single stock for a more mixed, less "designed" comparison.

### Regression Fit: R² and t-stats

In [ ]:
def factor_regression_stats(rets, factor_model):
    """Returns per-asset factor betas, R-squared, and t-stats, so we can see
    how much of each asset's return the model actually explains, and which
    factor exposures are statistically significant."""
    common_idx = rets.index.intersection(factor_model.index)
    rets = rets.loc[common_idx]
    factor_model = factor_model.loc[common_idx]

    factor_cols = factor_model.columns.drop('RF')
    rets_excess = rets - factor_model[['RF']].values
    X = factor_model[factor_cols].copy()
    X['Alpha'] = 1

    betas, rsquared, tstats = {}, {}, {}
    for col in rets_excess.columns:
        lm = sm.OLS(rets_excess[col], X).fit()
        betas[col] = lm.params
        rsquared[col] = lm.rsquared
        tstats[col] = lm.tvalues

    betas = pd.DataFrame(betas).round(3)
    rsquared = pd.Series(rsquared, name='R-squared').round(3)
    tstats = pd.DataFrame(tstats).round(2)

    return betas, rsquared, tstats

betas, rsquared, tstats = factor_regression_stats(rets, ff_6f)

In [ ]:
print('R-squared (share of variance explained by the 6 factors):')
rsquared

In [ ]:
print('t-stats per factor (rough rule of thumb: |t| > 2 ~ significant):')
tstats

### Factor Loadings, Visualized

In [ ]:
betas.drop('Alpha').T.plot(kind='bar', figsize=(10,5), title='Full-Sample Factor Loadings')
plt.ylabel('Beta')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()

## Rolling Factor Regression

In [ ]:
def rolling_factor_regression(rets, factor_model, roll_window=36):
    # common index
    common_idx = rets.index.intersection(factor_model.index)
    rets = rets.loc[common_idx]
    factor_model = factor_model.loc[common_idx]

    factor_cols = factor_model.columns.drop('RF')

    # prep data for regression
    rets_excess = rets - factor_model[['RF']].values
    X = factor_model[factor_cols]

    # rolling
    n_periods = rets.shape[0]
    windows = [(start, start + roll_window) for start in range(n_periods - roll_window + 1)]

    if n_periods <= roll_window:
        print('Sample smaller than roll window.')
        return

    # rolling regression
    results = {}
    for ticker in rets_excess:
        rows = []
        for win in windows:
            lm = sm.OLS(rets_excess[ticker].iloc[win[0]:win[1]], X.iloc[win[0]:win[1]]).fit()
            row = lm.params
            row['R2'] = lm.rsquared
            row['date'] = rets_excess.index[win[1] - 1]
            rows.append(row)
        results[ticker] = pd.DataFrame(rows).set_index('date')

    return results

In [ ]:
roll_window = 36

results = rolling_factor_regression(rets, ff_6f, roll_window)

for ticker in results:
    results[ticker].drop(columns='R2').plot(
        title=f'Rolling Factor Loadings ({roll_window} Months) on {ticker}', figsize=(10,5)
    )

### Rolling R²

How much of each asset's return the 6-factor model explains over time, instead of just over the full sample. Dips are worth a second look — e.g. around the 2020 COVID crash, factor relationships tend to break down as everything sells off together.

In [ ]:
plt.figure(figsize=(10,5))
for ticker in results:
    results[ticker]['R2'].plot(label=ticker)
plt.title(f'Rolling R² ({roll_window} Months)')
plt.ylabel('R-squared')
plt.legend()
plt.tight_layout()